# Tokenizer alignment và Experiment 1

In [1]:
from collections import Counter
import pandas as pd
from IPython.display import display
from transformers import AutoTokenizer
from ner_common import MODEL, align_labels, load_conll

data, names = load_conll()
tokenizer = AutoTokenizer.from_pretrained(MODEL)
sample = data['test'][0]
encoding = tokenizer(sample['tokens'], is_split_into_words=True)
aligned = align_labels(encoding, sample['ner_tags'])
table = pd.DataFrame({
    'token': tokenizer.convert_ids_to_tokens(encoding['input_ids']),
    'input_id': encoding['input_ids'],
    'word_id': encoding.word_ids(),
    'attention_mask': encoding['attention_mask'],
    'label': [names[i] if i != -100 else 'IGNORE(-100)' for i in aligned],
})
display(table)
assert len(encoding['input_ids']) == len(aligned) == len(encoding['attention_mask'])
assert all(aligned[i] == -100 for i, word_id in enumerate(encoding.word_ids()) if word_id is None)
assert sum(i != -100 for i in aligned) == len(sample['tokens'])

,token,input_id,word_id,attention_mask,label
0,[CLS],101,NaN,1,IGNORE(-100)
1,soccer,4715,0.0,1,O
2,-,1011,1.0,1,O
3,japan,2900,2.0,1,B-LOC
4,get,2131,3.0,1,O
5,lucky,5341,4.0,1,O
6,win,2663,5.0,1,O
7,",",1010,6.0,1,O
8,china,2859,7.0,1,B-PER
9,in,1999,8.0,1,O


In [2]:
import torch
from transformers import AutoModelForTokenClassification
from ner_common import ROOT

checkpoint = ROOT / 'checkpoint'
if checkpoint.exists():
    model = AutoModelForTokenClassification.from_pretrained(checkpoint, local_files_only=True)
    model.eval()
    with torch.no_grad():
        logits = model(input_ids=torch.tensor([encoding['input_ids']]),
                       attention_mask=torch.tensor([encoding['attention_mask']])).logits
    predicted = logits.argmax(-1)[0].tolist()
    table['prediction'] = [names[i] if aligned[j] != -100 else 'IGNORE(-100)'
                           for j, i in enumerate(predicted)]
    display(table[['token', 'word_id', 'label', 'prediction']])
    assert logits.shape == (1, len(aligned), len(names))
else:
    print('Prediction table is available after running ner_finetuning.ipynb.')

Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

,token,word_id,label,prediction
0,[CLS],NaN,IGNORE(-100),IGNORE(-100)
1,soccer,0.0,O,O
2,-,1.0,O,O
3,japan,2.0,B-LOC,B-LOC
4,get,3.0,O,O
5,lucky,4.0,O,O
6,win,5.0,O,O
7,",",6.0,O,O
8,china,7.0,B-PER,B-LOC
9,in,8.0,O,O


**Chiến lược A:** Chỉ subword đầu tiên của mỗi word nhận nhãn BIO. Special tokens và các mảnh tiếp theo nhận `-100`; chúng vẫn có `attention_mask=1` và tham gia self-attention, nhưng không đóng góp vào loss. Khi đánh giá, chỉ lấy logits tại vị trí nhãn khác `-100` để khôi phục đúng một nhãn trên mỗi word. Cách này giữ số gold labels nguyên gốc; nhược điểm là mô hình không nhận giám sát trực tiếp cho mảnh continuation.

In [3]:
# 10 câu ngắn 4–12 từ và 10 câu dài nhất, không trùng nhau.
ordered = sorted(range(len(data['test'])), key=lambda i: len(data['test'][i]['tokens']))
short_indices = [i for i in ordered if 4 <= len(data['test'][i]['tokens']) <= 12][:10]
long_indices = ordered[-10:]
indices = short_indices + long_indices
records = []
word_records = []
for index in indices:
    row = data['test'][index]
    encoded = tokenizer(row['tokens'], is_split_into_words=True)
    pieces = Counter(w for w in encoded.word_ids() if w is not None)
    words = len(row['tokens'])
    subword_tokens = sum(pieces.values())
    records.append({'sentence_id': row['id'], 'group': 'short' if index in short_indices else 'long',
                    'words': words, 'subword_tokens': subword_tokens,
                    'tokens_with_special': len(encoded['input_ids']),
                    'split_words': sum(n > 1 for n in pieces.values()),
                    'extra_subword_pieces': subword_tokens - words,
                    'R': subword_tokens / words})
    for word_id, word in enumerate(row['tokens']):
        word_records.append({'word': word, 'entity': row['ner_tags'][word_id] != 0,
                             'split': pieces[word_id] > 1, 'pieces': pieces[word_id]})
stats = pd.DataFrame(records)
word_stats = pd.DataFrame(word_records)
display(stats)
display(stats.groupby('group')[['words', 'subword_tokens', 'split_words', 'R']].mean())
display(word_stats.groupby('entity').agg(words=('word', 'size'), split_rate=('split', 'mean'), mean_pieces=('pieces', 'mean')))
display(word_stats[word_stats['split']].sort_values('pieces', ascending=False).head(15))
assert len(stats) == 20 and (stats['subword_tokens'] >= stats['words']).all()

,sentence_id,group,words,subword_tokens,tokens_with_special,split_words,extra_subword_pieces,R
0,24,short,4,7,9,2,3,1.750000
1,49,short,4,10,12,2,6,2.500000
2,81,short,4,6,8,1,2,1.500000
3,93,short,4,10,12,2,6,2.500000
4,94,short,4,5,7,1,1,1.250000
5,134,short,4,14,16,2,10,3.500000
6,141,short,4,4,6,0,0,1.000000
7,227,short,4,8,10,1,4,2.000000
8,230,short,4,7,9,2,3,1.750000
9,270,short,4,12,14,3,8,3.000000


,words,subword_tokens,split_words,R
group,,,,
long,74.2,97.2,14.4,1.330128
short,4.0,8.3,1.6,2.075000


,words,split_rate,mean_pieces
entity,,,
False,463,0.025918,1.088553
True,319,0.463950,1.727273


,word,entity,split,pieces
22,0.1-0-0-0,False,True,9
38,3-0-11-1,False,True,7
305,undersecretary-general,False,True,6
7,1996-12-06,False,True,5
31,1996-12-05,False,True,5
466,al-Boushi,True,True,5
262,Cristofoletto,True,True,5
15,1996-12-06,False,True,5
256,Giovanelli,True,True,4
244,Mazzariol,True,True,4


`R` loại special tokens để đo đúng số subword trên mỗi word. Trong 20 câu được chọn, câu ngắn có trung bình 4,0 words và R=2,075; câu dài có 74,2 words và R=1,330. Mẫu câu ngắn chứa nhiều tiêu đề, mã số và tên riêng nên tỷ lệ tách cao hơn dù ít từ hơn. Entity words có split rate 46,4%, non-entity words 2,6% trong mẫu; đây là quan sát của mẫu có chủ đích, không phải ước lượng toàn corpus. Các từ bị chia nhiều mảnh gồm `JosepGuardiola`, `Cristofoletto`, ngày tháng và từ ghép `undersecretary-general`. Chúng thường hiếm hoặc có hình thái khác từ vựng pretrained. Bảng thống kê phía trên giữ đủ 20 câu để kiểm tra từng con số.